# 12 infercnvpy qc

**Purpose:** Summarize and visualize infercnvpy results.

**Original analysis notebook:** `12_CNV_visual_QC_and_interpretation_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 28 — CNV visual QC and interpretation after infercnvpy

**Goal:** visually and quantitatively inspect infercnvpy CNV results across all CNV reference strategies.

This notebook does **not** make a final malignant call by itself.  
It checks whether the previous `CNV_low_or_reference_like` consensus is supported visually, or whether broad CNV patterns are present but missed by strict score thresholds.

Main checks:
- per-strategy run/result availability
- CNV score distributions
- transcriptomic UMAP colored by CNV score, CNV role, cluster, annotation
- infercnvpy chromosome heatmaps and summary heatmaps
- custom group-level CNV heatmaps from `adata.obsm["X_cnv"]` if available
- per-cluster/per-strategy visual interpretation table


In [ ]:
# =========================
# Imports and configuration
# =========================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt
import scipy.sparse as sp

try:
    import infercnvpy as cnv
    HAS_INFERCNVPY = True
except Exception as e:
    HAS_INFERCNVPY = False
    print("[WARNING] infercnvpy could not be imported:", repr(e))

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "interim"
FIGURES_DIR = PROJECT_DIR / "figures"

CNV_INPUT_DIR = RAW_DIR / "integrated" / "CNV_inputs"
CNV_RESULT_DIR = PROCESSED_DIR / "integrated" / "CNV_results"
CNV_FIG_DIR = FIGURES_DIR / "CNV_visual_QC_HVG8000_r0_6"

for d in [METADATA_DIR, CNV_RESULT_DIR, CNV_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_SUFFIX = "HVG8000_r0_6"

COMBINED_RUN_SUMMARY_TSV = METADATA_DIR / f"infercnvpy_combined_run_summary_{RUN_SUFFIX}.tsv"
COMBINED_CLUSTER_SUMMARY_TSV = METADATA_DIR / f"infercnvpy_combined_cluster_summary_{RUN_SUFFIX}.tsv"
CONSENSUS_CALLS_TSV = METADATA_DIR / f"infercnvpy_consensus_cluster_calls_{RUN_SUFFIX}.tsv"
INFERCNVPY_OUTPUT_MANIFEST_TSV = METADATA_DIR / f"infercnvpy_output_manifest_{RUN_SUFFIX}.tsv"

FIGURE_MANIFEST_TSV = METADATA_DIR / f"CNV_visual_QC_figure_manifest_{RUN_SUFFIX}.tsv"
STRATEGY_VISUAL_SUMMARY_TSV = METADATA_DIR / f"CNV_visual_QC_strategy_summary_{RUN_SUFFIX}.tsv"
CLUSTER_VISUAL_SUMMARY_TSV = METADATA_DIR / f"CNV_visual_QC_cluster_score_summary_{RUN_SUFFIX}.tsv"
CHROMOSOME_SIGNAL_TSV = METADATA_DIR / f"CNV_visual_QC_chromosome_signal_summary_{RUN_SUFFIX}.tsv"
INTERPRETATION_TSV = METADATA_DIR / f"CNV_visual_QC_interpretation_table_{RUN_SUFFIX}.tsv"
OUTPUT_MANIFEST_TSV = METADATA_DIR / f"CNV_visual_QC_output_manifest_{RUN_SUFFIX}.tsv"

print("PROJECT_DIR:", PROJECT_DIR)
print("CNV_RESULT_DIR:", CNV_RESULT_DIR)
print("CNV_FIG_DIR:", CNV_FIG_DIR)
print("HAS_INFERCNVPY:", HAS_INFERCNVPY)


In [ ]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def safe_read_tsv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path, sep="\t")
    print(f"[MISSING] {path}")
    return pd.DataFrame()

def sanitize_name(x):
    return str(x).replace("/", "_").replace(" ", "_").replace(".", "_").replace(":", "_")

def sparse_to_dense_array(x):
    if sp.issparse(x):
        return x.toarray()
    return np.asarray(x)

def finite_summary(x):
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {
            "min": np.nan, "q05": np.nan, "q25": np.nan, "median": np.nan,
            "mean": np.nan, "q75": np.nan, "q95": np.nan, "max": np.nan,
            "sd": np.nan
        }
    return {
        "min": float(np.min(x)),
        "q05": float(np.quantile(x, 0.05)),
        "q25": float(np.quantile(x, 0.25)),
        "median": float(np.median(x)),
        "mean": float(np.mean(x)),
        "q75": float(np.quantile(x, 0.75)),
        "q95": float(np.quantile(x, 0.95)),
        "max": float(np.max(x)),
        "sd": float(np.std(x)),
    }

def detect_cnv_rep(adata):
    # infercnvpy default: key_added='cnv' => adata.obsm['X_cnv']
    preferred = ["X_cnv", "cnv", "X_infercnv", "X_CNV"]
    for k in preferred:
        if k in adata.obsm:
            return k
    cnv_like = [k for k in adata.obsm.keys() if "cnv" in str(k).lower()]
    if cnv_like:
        return cnv_like[0]
    return None

def detect_cnv_score_col(adata):
    preferred = ["cnv_score", "CNV_score", "infercnvpy_cnv_score"]
    for c in preferred:
        if c in adata.obs.columns:
            return c
    score_like = [c for c in adata.obs.columns if ("cnv" in str(c).lower() and "score" in str(c).lower())]
    if score_like:
        return score_like[0]
    return None

def ensure_cnv_score(adata, cnv_rep_key, score_key="cnv_score_visual"):
    # Create a simple cell-level CNV amplitude score as mean(abs(X_cnv)) if cnv_score is missing.
    if score_key in adata.obs.columns:
        return score_key
    if cnv_rep_key is None:
        return None
    X = adata.obsm[cnv_rep_key]
    if sp.issparse(X):
        score = np.asarray(np.abs(X).mean(axis=1)).ravel()
    else:
        score = np.nanmean(np.abs(np.asarray(X)), axis=1)
    adata.obs[score_key] = score
    return score_key

def find_existing_col(adata, candidates):
    for c in candidates:
        if c in adata.obs.columns:
            return c
    return None

def find_strategy_h5ads():
    # Find infercnvpy result h5ad files robustly from manifests and folders.
    candidates = []

    manifest = safe_read_tsv(INFERCNVPY_OUTPUT_MANIFEST_TSV)
    if not manifest.empty:
        for col in manifest.columns:
            if "path" in col.lower() or "file" in col.lower() or "h5ad" in col.lower():
                for val in manifest[col].dropna().astype(str):
                    if val.endswith(".h5ad"):
                        p = Path(val)
                        if p.exists():
                            candidates.append(p)

    search_roots = [CNV_RESULT_DIR, CNV_INPUT_DIR, PROCESSED_DIR / "integrated", RAW_DIR / "integrated"]
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob(f"*infercnvpy*{RUN_SUFFIX}*.h5ad"))
            candidates.extend(root.rglob(f"*CNV*{RUN_SUFFIX}*infercnvpy*.h5ad"))
            candidates.extend(root.rglob(f"*CNV*{RUN_SUFFIX}*result*.h5ad"))

    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        if p.exists() and str(p) not in seen:
            unique.append(p)
            seen.add(str(p))

    return sorted(unique)

def infer_strategy_from_path(path):
    s = str(path)
    known = ["core_conservative", "core_plus_lymphoid", "extended_myeloid", "review_cycling", "review_ambiguous"]
    for k in known:
        if k in s:
            return k
    return Path(path).stem

def save_current_fig(path, dpi=180, bbox_inches="tight"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.savefig(path, dpi=dpi, bbox_inches=bbox_inches)
    print("[SAVED FIG]", path)
    plt.close()
    return str(path)


In [ ]:
# =========================
# Load previous summary tables
# =========================

combined_run_summary = safe_read_tsv(COMBINED_RUN_SUMMARY_TSV)
combined_cluster_summary = safe_read_tsv(COMBINED_CLUSTER_SUMMARY_TSV)
consensus_calls = safe_read_tsv(CONSENSUS_CALLS_TSV)
infer_manifest = safe_read_tsv(INFERCNVPY_OUTPUT_MANIFEST_TSV)

print("combined_run_summary:", combined_run_summary.shape)
print("combined_cluster_summary:", combined_cluster_summary.shape)
print("consensus_calls:", consensus_calls.shape)
print("infer_manifest:", infer_manifest.shape)

if not consensus_calls.empty:
    display(consensus_calls)

if not combined_run_summary.empty:
    display(combined_run_summary)


In [ ]:
# =========================
# Discover infercnvpy result h5ad files
# =========================

strategy_h5ads = find_strategy_h5ads()

discovery = pd.DataFrame({
    "path": [str(p) for p in strategy_h5ads],
    "strategy_guess": [infer_strategy_from_path(p) for p in strategy_h5ads],
    "exists": [p.exists() for p in strategy_h5ads],
    "size_MB": [round(p.stat().st_size / 1024**2, 2) for p in strategy_h5ads],
})

display(discovery)

if discovery.empty:
    raise FileNotFoundError(
        "No infercnvpy result h5ad files found. "
        "Check the previous infercnvpy output manifest and CNV_results folder."
    )


In [ ]:
# =========================
# Inspect each infercnvpy result object
# =========================

inspection_rows = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    print("\n" + "="*90)
    print(strategy)
    print(p)

    adata_cnv = sc.read_h5ad(p)
    cnv_rep = detect_cnv_rep(adata_cnv)
    score_col = detect_cnv_score_col(adata_cnv)
    score_col_final = score_col if score_col is not None else ensure_cnv_score(adata_cnv, cnv_rep)

    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    annotation_col = find_existing_col(adata_cnv, ["annotation_marker_based_HVG8000_r0_6", "annotation_marker_based", "annotation"])
    cnv_leiden_col = find_existing_col(adata_cnv, ["cnv_leiden", "CNV_leiden", "infercnvpy_leiden"])

    if score_col_final is not None:
        score_stats = finite_summary(adata_cnv.obs[score_col_final].astype(float).values)
    else:
        score_stats = finite_summary([])

    inspection_rows.append({
        "strategy": strategy,
        "h5ad_path": str(p),
        "n_cells": adata_cnv.n_obs,
        "n_genes": adata_cnv.n_vars,
        "obsm_keys": ";".join(map(str, adata_cnv.obsm.keys())),
        "obs_columns_n": len(adata_cnv.obs.columns),
        "var_columns": ";".join(map(str, adata_cnv.var.columns)),
        "cnv_rep_key": cnv_rep,
        "cnv_score_col": score_col_final,
        "role_col": role_col,
        "cluster_col": cluster_col,
        "annotation_col": annotation_col,
        "cnv_leiden_col": cnv_leiden_col,
        **{f"cnv_score_{k}": v for k, v in score_stats.items()},
    })

inspection_df = pd.DataFrame(inspection_rows)
write_tsv_replace(inspection_df, STRATEGY_VISUAL_SUMMARY_TSV)
display(inspection_df)


In [ ]:
# =========================
# Plot transcriptomic UMAPs per strategy
# =========================

figure_records = []
cluster_score_records = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)

    cnv_rep = detect_cnv_rep(adata_cnv)
    score_col = detect_cnv_score_col(adata_cnv)
    if score_col is None:
        score_col = ensure_cnv_score(adata_cnv, cnv_rep)

    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    annotation_col = find_existing_col(adata_cnv, ["annotation_marker_based_HVG8000_r0_6", "annotation_marker_based", "annotation"])
    cnv_leiden_col = find_existing_col(adata_cnv, ["cnv_leiden", "CNV_leiden", "infercnvpy_leiden"])

    if "X_umap" not in adata_cnv.obsm:
        print(f"[SKIP UMAP] X_umap not found for {strategy}")
        continue

    colors = [c for c in [score_col, role_col, cluster_col, annotation_col, cnv_leiden_col] if c is not None and c in adata_cnv.obs.columns]
    if len(colors) == 0:
        continue

    for color in colors:
        try:
            sc.pl.umap(
                adata_cnv,
                color=color,
                show=False,
                frameon=False,
                size=4 if adata_cnv.n_obs < 50000 else 2,
                title=f"{strategy} — {color}",
            )
            out = CNV_FIG_DIR / f"UMAP_{sanitize_name(strategy)}_{sanitize_name(color)}.png"
            save_current_fig(out)
            figure_records.append({
                "strategy": strategy,
                "figure_type": "transcriptomic_umap",
                "color": color,
                "path": str(out),
            })
        except Exception as e:
            print(f"[WARNING] UMAP failed for {strategy} {color}: {repr(e)}")

    if cluster_col is not None and score_col is not None:
        tmp = adata_cnv.obs[[cluster_col, score_col]].copy()
        tmp[score_col] = tmp[score_col].astype(float)
        grp = tmp.groupby(cluster_col, observed=True)[score_col].agg(
            n_cells="size",
            cnv_score_mean="mean",
            cnv_score_median="median",
            cnv_score_q75=lambda x: np.quantile(x, 0.75),
            cnv_score_q95=lambda x: np.quantile(x, 0.95),
            cnv_score_max="max",
        ).reset_index().rename(columns={cluster_col: "cluster"})
        grp["strategy"] = strategy
        cluster_score_records.append(grp)

if len(cluster_score_records) > 0:
    cluster_score_df = pd.concat(cluster_score_records, ignore_index=True)
else:
    cluster_score_df = pd.DataFrame()

write_tsv_replace(cluster_score_df, CLUSTER_VISUAL_SUMMARY_TSV)
display(cluster_score_df.head(30))


In [ ]:
# =========================
# infercnvpy chromosome heatmaps
# =========================

if not HAS_INFERCNVPY:
    print("[SKIP] infercnvpy unavailable.")
else:
    for p in strategy_h5ads:
        strategy = infer_strategy_from_path(p)
        adata_cnv = sc.read_h5ad(p)

        cnv_rep_key = detect_cnv_rep(adata_cnv)
        if cnv_rep_key is None:
            print(f"[SKIP] no CNV representation for {strategy}")
            continue

        if cnv_rep_key.startswith("X_"):
            use_rep = cnv_rep_key[2:]
        else:
            use_rep = cnv_rep_key

        group_candidates = [
            "cnv_leiden",
            "leiden_scvi_main",
            "CNV_role",
            "cnv_role",
            "annotation_marker_based_HVG8000_r0_6",
        ]
        group_cols = [g for g in group_candidates if g in adata_cnv.obs.columns]

        for groupby in group_cols:
            try:
                cnv.pl.chromosome_heatmap(
                    adata_cnv,
                    groupby=groupby,
                    use_rep=use_rep,
                    show=False,
                )
                out = CNV_FIG_DIR / f"infercnvpy_chromosome_heatmap_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
                save_current_fig(out, dpi=160)
                figure_records.append({
                    "strategy": strategy,
                    "figure_type": "infercnvpy_chromosome_heatmap",
                    "groupby": groupby,
                    "path": str(out),
                })
            except Exception as e:
                print(f"[WARNING] chromosome_heatmap failed for {strategy} groupby={groupby}: {repr(e)}")

        for groupby in group_cols:
            try:
                cnv.pl.chromosome_heatmap_summary(
                    adata_cnv,
                    groupby=groupby,
                    use_rep=use_rep,
                    show=False,
                )
                out = CNV_FIG_DIR / f"infercnvpy_chromosome_heatmap_summary_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
                save_current_fig(out, dpi=160)
                figure_records.append({
                    "strategy": strategy,
                    "figure_type": "infercnvpy_chromosome_heatmap_summary",
                    "groupby": groupby,
                    "path": str(out),
                })
            except Exception as e:
                print(f"[WARNING] chromosome_heatmap_summary failed for {strategy} groupby={groupby}: {repr(e)}")


In [ ]:
# =========================
# Custom group-level CNV heatmaps from X_cnv
# =========================

def group_mean_matrix(adata, rep_key, groupby, max_groups=80):
    if rep_key is None or groupby not in adata.obs.columns:
        return None, None

    X = adata.obsm[rep_key]
    groups = adata.obs[groupby].astype(str)
    group_order = groups.value_counts().index.tolist()
    if len(group_order) > max_groups:
        group_order = group_order[:max_groups]

    rows = []
    names = []
    for g in group_order:
        idx = np.where(groups.values == g)[0]
        if len(idx) == 0:
            continue
        Xi = X[idx]
        if sp.issparse(Xi):
            mean = np.asarray(Xi.mean(axis=0)).ravel()
        else:
            mean = np.nanmean(np.asarray(Xi), axis=0)
        rows.append(mean)
        names.append(g)

    if len(rows) == 0:
        return None, None

    return np.vstack(rows), names

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)
    rep_key = detect_cnv_rep(adata_cnv)

    if rep_key is None:
        continue

    groupby_candidates = [
        "leiden_scvi_main",
        "cnv_leiden",
        "CNV_role",
        "cnv_role",
        "annotation_marker_based_HVG8000_r0_6",
    ]
    groupby_candidates = [g for g in groupby_candidates if g in adata_cnv.obs.columns]

    for groupby in groupby_candidates:
        M, names = group_mean_matrix(adata_cnv, rep_key, groupby)
        if M is None:
            continue

        vmax = np.nanquantile(np.abs(M), 0.98)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        plt.figure(figsize=(16, max(4, 0.28 * len(names))))
        plt.imshow(M, aspect="auto", cmap="bwr", vmin=-vmax, vmax=vmax, interpolation="nearest")
        plt.yticks(np.arange(len(names)), names, fontsize=8)
        plt.xlabel("CNV genomic window/bin")
        plt.ylabel(groupby)
        plt.title(f"{strategy} — mean CNV profile by {groupby}")
        plt.colorbar(label="mean CNV signal")
        out = CNV_FIG_DIR / f"custom_mean_CNV_heatmap_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
        save_current_fig(out, dpi=160)
        figure_records.append({
            "strategy": strategy,
            "figure_type": "custom_mean_CNV_heatmap",
            "groupby": groupby,
            "path": str(out),
        })


In [ ]:
# =========================
# Try chromosome-level summaries from CNV bins
# =========================

def extract_window_chromosomes(adata, rep_key):
    """Best-effort extraction of chromosome label for each CNV window."""
    n_bins = adata.obsm[rep_key].shape[1] if rep_key is not None else None
    if n_bins is None:
        return None

    candidate_uns_keys = []
    if rep_key.startswith("X_"):
        candidate_uns_keys.append(rep_key[2:])
    candidate_uns_keys.append(rep_key)
    candidate_uns_keys.extend([k for k in adata.uns.keys() if "cnv" in str(k).lower()])

    for uk in candidate_uns_keys:
        if uk not in adata.uns:
            continue
        obj = adata.uns[uk]

        if isinstance(obj, dict):
            for key in ["chr", "chromosome", "chromosomes", "chr_pos", "chromosome_pos", "window_chromosomes"]:
                if key in obj:
                    val = obj[key]
                    try:
                        arr = np.asarray(val)
                        if arr.ndim == 1 and len(arr) == n_bins:
                            return arr.astype(str)
                    except Exception:
                        pass

            for key, val in obj.items():
                try:
                    if isinstance(val, pd.DataFrame):
                        for col in ["chromosome", "chr", "cnv_chr"]:
                            if col in val.columns and len(val) == n_bins:
                                return val[col].astype(str).values
                except Exception:
                    pass

    return None

chrom_records = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)
    rep_key = detect_cnv_rep(adata_cnv)
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])

    if rep_key is None:
        continue

    chroms = extract_window_chromosomes(adata_cnv, rep_key)
    if chroms is None:
        print(f"[INFO] Could not extract chromosome per CNV window for {strategy}; skipping chromosome-level signal table.")
        continue

    X = adata_cnv.obsm[rep_key]
    group_cols = [c for c in [cluster_col, role_col, "cnv_leiden"] if c is not None and c in adata_cnv.obs.columns]

    for group_col in group_cols:
        groups = adata_cnv.obs[group_col].astype(str)

        for group in sorted(groups.unique(), key=lambda x: str(x)):
            idx = np.where(groups.values == group)[0]
            if len(idx) == 0:
                continue
            Xi = X[idx]
            if sp.issparse(Xi):
                mean_bin = np.asarray(Xi.mean(axis=0)).ravel()
            else:
                mean_bin = np.nanmean(np.asarray(Xi), axis=0)

            for chrom in pd.unique(chroms):
                j = np.where(chroms == chrom)[0]
                if len(j) == 0:
                    continue
                vals = mean_bin[j]
                chrom_records.append({
                    "strategy": strategy,
                    "groupby": group_col,
                    "group": group,
                    "chromosome": chrom,
                    "n_windows": int(len(j)),
                    "mean_cnv_signal": float(np.nanmean(vals)),
                    "median_cnv_signal": float(np.nanmedian(vals)),
                    "mean_abs_cnv_signal": float(np.nanmean(np.abs(vals))),
                    "max_abs_cnv_signal": float(np.nanmax(np.abs(vals))),
                })

chrom_df = pd.DataFrame(chrom_records)
write_tsv_replace(chrom_df, CHROMOSOME_SIGNAL_TSV)
display(chrom_df.head(30))


In [ ]:
# =========================
# Build interpretation table
# =========================

if not cluster_score_df.empty:
    score_pivot = cluster_score_df.copy()
    score_pivot["cluster"] = score_pivot["cluster"].astype(str)

    agg = score_pivot.groupby("cluster", observed=True).agg(
        n_strategies_observed=("strategy", "nunique"),
        max_cluster_median_cnv_score=("cnv_score_median", "max"),
        median_cluster_median_cnv_score=("cnv_score_median", "median"),
        max_cluster_q95_cnv_score=("cnv_score_q95", "max"),
        max_cluster_max_cnv_score=("cnv_score_max", "max"),
    ).reset_index()
else:
    agg = pd.DataFrame(columns=[
        "cluster", "n_strategies_observed", "max_cluster_median_cnv_score",
        "median_cluster_median_cnv_score", "max_cluster_q95_cnv_score", "max_cluster_max_cnv_score"
    ])

if not consensus_calls.empty:
    cc = consensus_calls.copy()
    if "cluster" not in cc.columns:
        possible = [c for c in cc.columns if "cluster" in c.lower()]
        if possible:
            cc = cc.rename(columns={possible[0]: "cluster"})
    cc["cluster"] = cc["cluster"].astype(str)
    interp = cc.merge(agg, on="cluster", how="left")
else:
    interp = agg.copy()

def interpret_row(row):
    max_med = row.get("max_cluster_median_cnv_score", np.nan)
    max_q95 = row.get("max_cluster_q95_cnv_score", np.nan)
    consensus = str(row.get("consensus_call", row.get("CNV_consensus_call", ""))).lower()

    if "high" in consensus:
        return "visual_review_priority_high_consensus_cnv_high"
    if np.isfinite(max_med) and max_med >= 0.15:
        return "visual_review_priority_possible_broad_cnv"
    if np.isfinite(max_q95) and max_q95 >= 0.30:
        return "visual_review_priority_subpopulation_possible_cnv"
    if "low" in consensus or "reference" in consensus:
        return "currently_reference_like_by_score_visual_check_needed"
    return "insufficient_information_visual_check_needed"

if not interp.empty:
    interp["visual_interpretation_flag"] = interp.apply(interpret_row, axis=1)

write_tsv_replace(interp, INTERPRETATION_TSV)
display(interp)


In [ ]:
# =========================
# Save figure and output manifests
# =========================

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, FIGURE_MANIFEST_TSV)

output_manifest = pd.DataFrame([
    {"file_type": "table", "description": "strategy-level visual QC summary", "path": str(STRATEGY_VISUAL_SUMMARY_TSV), "exists": STRATEGY_VISUAL_SUMMARY_TSV.exists()},
    {"file_type": "table", "description": "cluster-level CNV score summary across strategies", "path": str(CLUSTER_VISUAL_SUMMARY_TSV), "exists": CLUSTER_VISUAL_SUMMARY_TSV.exists()},
    {"file_type": "table", "description": "chromosome-level CNV signal summary when chromosome-bin metadata is available", "path": str(CHROMOSOME_SIGNAL_TSV), "exists": CHROMOSOME_SIGNAL_TSV.exists()},
    {"file_type": "table", "description": "cautious CNV visual interpretation table", "path": str(INTERPRETATION_TSV), "exists": INTERPRETATION_TSV.exists()},
    {"file_type": "table", "description": "figure manifest", "path": str(FIGURE_MANIFEST_TSV), "exists": FIGURE_MANIFEST_TSV.exists()},
    {"file_type": "folder", "description": "CNV visual QC figures", "path": str(CNV_FIG_DIR), "exists": CNV_FIG_DIR.exists()},
])
write_tsv_replace(output_manifest, OUTPUT_MANIFEST_TSV)
display(output_manifest)

print("\nImportant figure folder:")
print(CNV_FIG_DIR)


## What to send back

After running this notebook, send these TSV files:

```text
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_strategy_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_cluster_score_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_chromosome_signal_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_interpretation_table_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_figure_manifest_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/CNV_visual_QC_output_manifest_HVG8000_r0_6.tsv
```

Also send screenshots of the most informative heatmaps/UMAPs if the TSVs suggest any cluster has possible CNV signal.
